# Notebook 1: Tiền xử lý dữ liệu cho khai phá luật kết hợp định lượng

Notebook này chuẩn bị dữ liệu cho hai kịch bản thực nghiệm ở Chương 5:

- **Baseline (Post-processing):** chạy FP-Growth trước, sau đó lọc các mẫu có `avg(price) > 20`.
- **Proposed (In-mining Pruning):** cắt tỉa trực tiếp trong quá trình sinh mẫu khi `avg(price) > 20`.

Mục tiêu của notebook này là đọc dataset Market Basket Analysis, làm sạch dữ liệu theo yêu cầu, gom nhóm các mặt hàng theo mã hóa đơn `BillNo`, tạo danh sách giao dịch, tạo bảng giá đại diện cho từng mặt hàng, hiển thị đầy đủ các thống kê cần dùng trong báo cáo, và lưu kết quả tiền xử lý cho các notebook tiếp theo.

## 1. Cấu hình môi trường

Notebook được thiết kế để chạy được trên cả máy local:

- Khi chạy local, notebook tự tìm dataset trong thư mục dự án.
- Khi chạy trên Colab, nếu chưa tìm thấy dataset, notebook sẽ yêu cầu upload file CSV hoặc Excel.
- Có thể đổi đường dẫn dataset bằng biến môi trường `MARKET_BASKET_DATASET_PATH`.
- Có thể đổi thư mục đầu ra bằng biến môi trường `OUTPUT_DIR`.

In [1]:
import importlib.util
import json
import os
import pickle
import subprocess
import sys
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    """Cài thư viện nếu môi trường hiện tại chưa có.

    Khi chạy trong Codex Desktop/local hiện tại, hàm cũng thử dùng thư mục thư viện
    Python đi kèm Codex trước khi gọi pip. Trên Colab/Jupyter thông thường, pandas
    thường đã có sẵn nên bước này không phát sinh cài đặt.
    """
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        bundled_packages = Path.home() / ".cache" / "codex-runtimes" / "codex-primary-runtime" / "dependencies" / "python"
        if bundled_packages.exists() and str(bundled_packages) not in sys.path:
            sys.path.insert(0, str(bundled_packages))

    if importlib.util.find_spec(import_name) is None:
        print(f"Thiếu thư viện '{import_name}'. Đang cài đặt gói '{pip_name}'...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])


ensure_package("pandas")

import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# =========================
# CẤU HÌNH LINH HOẠT
# =========================
# Có thể chỉnh bằng biến môi trường hoặc sửa trực tiếp tại đây khi chạy trên Colab/local.
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
DATASET_PATH_OVERRIDE = os.environ.get("MARKET_BASKET_DATASET_PATH", "").strip()
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", PROJECT_ROOT / "outputs")).resolve()

# Tên cột được cấu hình để notebook có thể tái sử dụng với dataset cùng cấu trúc nhưng đổi tên file.
TRANSACTION_ID_COLUMN = os.environ.get("TRANSACTION_ID_COLUMN", "BillNo")
ITEM_COLUMN = os.environ.get("ITEM_COLUMN", "Itemname")
PRICE_COLUMN = os.environ.get("PRICE_COLUMN", "Price")
QUANTITY_COLUMN = os.environ.get("QUANTITY_COLUMN", "Quantity")
REQUIRED_COLUMNS = [TRANSACTION_ID_COLUMN, ITEM_COLUMN, PRICE_COLUMN]

# Cách lấy giá đại diện cho từng item: median, mean, min hoặc max.
PRICE_AGG_METHOD = os.environ.get("PRICE_AGG_METHOD", "median").lower()
if PRICE_AGG_METHOD not in {"median", "mean", "min", "max"}:
    raise ValueError("PRICE_AGG_METHOD phải là một trong: median, mean, min, max")

TRANSACTION_ITEM_SEPARATOR = os.environ.get("TRANSACTION_ITEM_SEPARATOR", " | ")
ARTIFACT_FILES = {
    "transactions_pickle": os.environ.get("TRANSACTIONS_PICKLE", "transactions.pkl"),
    "transactions_csv": os.environ.get("TRANSACTIONS_CSV", "transactions.csv"),
    "item_price_pickle": os.environ.get("ITEM_PRICE_PICKLE", "item_price.pkl"),
    "item_price_csv": os.environ.get("ITEM_PRICE_CSV", "item_price.csv"),
    "summary_json": os.environ.get("PREPROCESSING_SUMMARY_JSON", "preprocessing_summary.json"),
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 80)

print("Môi trường chạy:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Thư mục làm việc:", PROJECT_ROOT)
print("Thư mục lưu kết quả:", OUTPUT_DIR)
print("Cột mã giao dịch:", TRANSACTION_ID_COLUMN)
print("Cột tên mặt hàng:", ITEM_COLUMN)
print("Cột giá:", PRICE_COLUMN)
print("Cách lấy giá đại diện:", PRICE_AGG_METHOD)

Môi trường chạy: Local/Jupyter
Thư mục làm việc: H:\chuong_trinh_hoc_UEH\mon_hoc_ki_6\data_mining\Lab\Data_Mining_Labs\Mining_Quantitative_Association_Rules_Lab
Thư mục lưu kết quả: H:\chuong_trinh_hoc_UEH\mon_hoc_ki_6\data_mining\Lab\Data_Mining_Labs\Mining_Quantitative_Association_Rules_Lab\outputs
Cột mã giao dịch: BillNo
Cột tên mặt hàng: Itemname
Cột giá: Price
Cách lấy giá đại diện: median


## 2. Tìm và đọc tập dữ liệu

Dataset được ưu tiên đọc từ file CSV vì tốc độ nhanh hơn. Nếu không có CSV, notebook sẽ đọc file Excel. Notebook tự dò separator phổ biến của CSV (`;`, `,`, tab) và tự chuẩn hóa cột giá, nên không phụ thuộc cứng vào đúng một tên file hay đúng một kiểu dấu thập phân.

In [2]:
SUPPORTED_DATASET_EXTENSIONS = {".csv", ".xlsx", ".xls"}


def infer_csv_separator(dataset_path: Path) -> str:
    """Tự dò separator CSV từ dòng đầu tiên."""
    with dataset_path.open("r", encoding="utf-8-sig", errors="ignore") as f:
        first_line = f.readline()
    candidates = [";", ",", "\t", "|"]
    return max(candidates, key=lambda sep: first_line.count(sep))


def read_preview_columns(dataset_path: Path):
    """Đọc nhanh vài dòng để kiểm tra dataset có đủ cột bắt buộc hay không."""
    suffix = dataset_path.suffix.lower()
    if suffix == ".csv":
        sep = infer_csv_separator(dataset_path)
        preview = pd.read_csv(dataset_path, sep=sep, nrows=5, dtype="string")
    elif suffix in {".xlsx", ".xls"}:
        ensure_package("openpyxl")
        preview = pd.read_excel(dataset_path, nrows=5, dtype="string")
    else:
        return []
    return list(preview.columns)


def looks_like_target_dataset(dataset_path: Path) -> bool:
    """Kiểm tra file có đủ các cột được cấu hình cho bài toán hay không."""
    try:
        columns = read_preview_columns(dataset_path)
    except Exception:
        return False
    return set(REQUIRED_COLUMNS).issubset(set(columns))


def candidate_dataset_files(project_root: Path):
    """Sinh danh sách file dữ liệu ứng viên, không phụ thuộc vào một tên file cố định."""
    if DATASET_PATH_OVERRIDE:
        yield Path(DATASET_PATH_OVERRIDE).expanduser()

    search_roots = [project_root]
    if IN_COLAB:
        search_roots.append(Path("/content"))

    seen = set()
    candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        for suffix in SUPPORTED_DATASET_EXTENSIONS:
            candidates.extend(root.glob(f"*{suffix}"))
            candidates.extend(root.glob(f"*/*{suffix}"))
            candidates.extend(root.glob(f"**/*{suffix}"))

    def sort_key(path):
        suffix_rank = 0 if path.suffix.lower() == ".csv" else 1
        return (suffix_rank, len(path.parts), path.name.lower())

    for path in sorted(candidates, key=sort_key):
        resolved = path.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if OUTPUT_DIR in resolved.parents:
            continue
        yield resolved


def find_dataset(project_root: Path) -> Path:
    """Tìm dataset dựa trên phần mở rộng và bộ cột bắt buộc đã cấu hình."""
    for path in candidate_dataset_files(project_root):
        if path.exists() and path.suffix.lower() in SUPPORTED_DATASET_EXTENSIONS and looks_like_target_dataset(path):
            return path

    if IN_COLAB:
        from google.colab import files  # type: ignore
        print("Không tìm thấy dataset phù hợp. Vui lòng upload file CSV hoặc Excel có đủ các cột đã cấu hình.")
        uploaded = files.upload()
        for filename in uploaded.keys():
            uploaded_path = Path("/content") / filename
            if uploaded_path.suffix.lower() in SUPPORTED_DATASET_EXTENSIONS and looks_like_target_dataset(uploaded_path):
                return uploaded_path

    raise FileNotFoundError(
        "Không tìm thấy dataset phù hợp. Hãy đặt file CSV/Excel có đủ các cột "
        f"{REQUIRED_COLUMNS} trong thư mục dự án, hoặc đặt biến MARKET_BASKET_DATASET_PATH."
    )


def read_market_basket_dataset(dataset_path: Path) -> pd.DataFrame:
    """Đọc dataset Market Basket Analysis từ CSV hoặc Excel."""
    suffix = dataset_path.suffix.lower()
    if suffix == ".csv":
        sep = infer_csv_separator(dataset_path)
        df = pd.read_csv(dataset_path, sep=sep, dtype="string")
    elif suffix in {".xlsx", ".xls"}:
        ensure_package("openpyxl")
        df = pd.read_excel(dataset_path, dtype="string")
    else:
        raise ValueError(f"Định dạng file không được hỗ trợ: {dataset_path.suffix}")

    return df


dataset_path = find_dataset(PROJECT_ROOT)
raw_df = read_market_basket_dataset(dataset_path)

print("Đường dẫn dataset:", dataset_path)
print("Kích thước dữ liệu gốc:", raw_df.shape)
display(raw_df.head(10))

Đường dẫn dataset: H:\chuong_trinh_hoc_UEH\mon_hoc_ki_6\data_mining\Lab\Data_Mining_Labs\Mining_Quantitative_Association_Rules_Lab\market_basket_analysis_dataset\Assignment-1_Data.csv
Kích thước dữ liệu gốc: (522064, 7)


,BillNo,Itemname,Quantity,Date,Price,CustomerID,Country
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,01.12.2010 08:26,"2,55",17850,United Kingdom
1,536365,WHITE METAL LANTERN,6,01.12.2010 08:26,"3,39",17850,United Kingdom
2,536365,CREAM CUPID HEARTS COAT HANGER,8,01.12.2010 08:26,"2,75",17850,United Kingdom
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,01.12.2010 08:26,"3,39",17850,United Kingdom
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6,01.12.2010 08:26,"3,39",17850,United Kingdom
5,536365,SET 7 BABUSHKA NESTING BOXES,2,01.12.2010 08:26,"7,65",17850,United Kingdom
6,536365,GLASS STAR FROSTED T-LIGHT HOLDER,6,01.12.2010 08:26,"4,25",17850,United Kingdom
7,536366,HAND WARMER UNION JACK,6,01.12.2010 08:28,"1,85",17850,United Kingdom
8,536366,HAND WARMER RED POLKA DOT,6,01.12.2010 08:28,"1,85",17850,United Kingdom
9,536367,ASSORTED COLOUR BIRD ORNAMENT,32,01.12.2010 08:34,"1,69",13047,United Kingdom


## 3. Kiểm tra cấu trúc và chất lượng dữ liệu gốc

Bước này hiển thị các cột, kiểu dữ liệu, số lượng giá trị thiếu và một vài thống kê ban đầu. Các thông tin này giúp xác nhận dataset đã được đọc đúng định dạng trước khi làm sạch.

In [3]:
required_columns = REQUIRED_COLUMNS
missing_required_columns = [col for col in required_columns if col not in raw_df.columns]
if missing_required_columns:
    raise KeyError(f"Dataset thiếu các cột bắt buộc: {missing_required_columns}")

# Chuẩn hóa kiểu dữ liệu cho các cột quan trọng.
raw_df[TRANSACTION_ID_COLUMN] = raw_df[TRANSACTION_ID_COLUMN].astype("string")
raw_df[ITEM_COLUMN] = raw_df[ITEM_COLUMN].astype("string")
raw_df[PRICE_COLUMN] = pd.to_numeric(
    raw_df[PRICE_COLUMN].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)
if QUANTITY_COLUMN in raw_df.columns:
    raw_df[QUANTITY_COLUMN] = pd.to_numeric(
        raw_df[QUANTITY_COLUMN].astype("string").str.replace(",", ".", regex=False),
        errors="coerce",
    )

schema_df = pd.DataFrame({
    "Cột": raw_df.columns,
    "Kiểu dữ liệu": [str(raw_df[col].dtype) for col in raw_df.columns],
    "Số giá trị thiếu": [int(raw_df[col].isna().sum()) for col in raw_df.columns],
    "Tỷ lệ thiếu (%)": [round(float(raw_df[col].isna().mean() * 100), 4) for col in raw_df.columns],
})

price_describe = raw_df[PRICE_COLUMN].describe().to_frame(name=PRICE_COLUMN).T

print("Thông tin cột và giá trị thiếu:")
display(schema_df)

print(f"Thống kê mô tả ban đầu của cột {PRICE_COLUMN}:")
display(price_describe)

if QUANTITY_COLUMN in raw_df.columns:
    quantity_describe = raw_df[QUANTITY_COLUMN].describe().to_frame(name=QUANTITY_COLUMN).T
    print(f"Thống kê mô tả ban đầu của cột {QUANTITY_COLUMN}:")
    display(quantity_describe)

Thông tin cột và giá trị thiếu:


,Cột,Kiểu dữ liệu,Số giá trị thiếu,Tỷ lệ thiếu (%)
0,BillNo,string,0,0.0000
1,Itemname,string,1455,0.2787
2,Quantity,Int64,0,0.0000
3,Date,string,0,0.0000
4,Price,Float64,0,0.0000
5,CustomerID,string,134041,25.6752
6,Country,string,0,0.0000


Thống kê mô tả ban đầu của cột Price:


,count,mean,std,min,25%,50%,75%,max
Price,522064.0,3.826801,41.900599,-11062.06,1.25,2.08,4.13,13541.33


Thống kê mô tả ban đầu của cột Quantity:


,count,mean,std,min,25%,50%,75%,max
Quantity,522064.0,10.090435,161.110525,-9600.0,1.0,3.0,10.0,80995.0
